# Submission v7b — Fix Class 1 Overconfidence

## What happened in v7
LOPO jumped to **0.5123** (best ever, up from 0.4288) — the absolute+change+HRV features work.

**But**: prediction distribution is broken — 56.5% class 1 vs 8.1% in training.
The HRV features (especially LF/HF) are noisy due to only 1Hz effective HR resolution,
causing false class 1 signals on test data where we have no training baseline.

## Fixes in v7b
1. **Remove HRV frequency features** (lf_hf, lf_nu, hf_nu) — LF/HF from 1Hz downsampled BPM is unreliable
2. **Keep HRV time-domain** (sdnn, rmssd, pnn25, pnn50, mean_rr, cv_rr) — these are robust at 1Hz
3. **Lower class 1 weight** — reduce from 4.12 to 2.5 to prevent class 1 overconfidence
4. **Add prior probability calibration** — after soft-vote, rescale probabilities toward training prior


In [1]:
%pip install lightgbm scikit-learn pandas numpy scipy -q


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from scipy import stats as spstats
from sklearn.model_selection import StratifiedKFold, LeaveOneGroupOut
from sklearn.metrics import balanced_accuracy_score
from sklearn.impute import SimpleImputer
import lightgbm as lgb
from collections import Counter

TRAIN_DATA  = pd.read_csv('train-sensor.csv')
TRAIN_LABEL = pd.read_csv('train-label.csv')
TEST_DATA   = pd.read_csv('test-sensor.csv')
TEST_LABEL  = pd.read_csv('test-label.csv')

TRAIN_LABEL['timestamp'] = TRAIN_LABEL['timestamp'].astype(float)
TEST_LABEL['timestamp']  = TEST_LABEL['timestamp'].astype(str).str.strip().astype(float)

print('TRAIN_DATA shape :', TRAIN_DATA.shape)
print('TRAIN_LABEL shape:', TRAIN_LABEL.shape)
print('TEST_DATA shape  :', TEST_DATA.shape)
print('TEST_LABEL shape :', TEST_LABEL.shape)
print()
print('Stress distribution:')
print(TRAIN_LABEL['stress'].value_counts().sort_index())


TRAIN_DATA shape : (4694400, 8)
TRAIN_LABEL shape: (815, 4)
TEST_DATA shape  : (5921280, 8)
TEST_LABEL shape : (1028, 4)

Stress distribution:
stress
0.0    162
1.0     66
2.0    587
Name: count, dtype: int64


In [3]:
SENSOR_COLS = ['accel_x','accel_y','accel_z','eda','heart_rate','temperature']
WINDOW_MS = 180_000
HALF_MS   =  90_000
THIRD_MS  =  60_000

def hrv_time_domain(bpm_series):
    """Time-domain HRV only — robust at 1Hz. No LF/HF (unreliable from BPM downsampled)."""
    f = {}
    bpm = bpm_series.dropna().values.astype(float)
    if len(bpm) < 10:
        for k in ['sdnn','rmssd','pnn25','pnn50','mean_rr','cv_rr']:
            f['hrv_' + k] = np.nan
        return f
    bpm_1hz = bpm[::32] if len(bpm) >= 32 else bpm
    rr = 60000.0 / np.clip(bpm_1hz, 30, 220)
    rr_diff = np.diff(rr)
    f['hrv_sdnn']    = float(np.std(rr))
    f['hrv_rmssd']   = float(np.sqrt(np.mean(rr_diff**2))) if len(rr_diff)>0 else 0.
    f['hrv_pnn25']   = float(np.mean(np.abs(rr_diff)>25))*100 if len(rr_diff)>0 else 0.
    f['hrv_pnn50']   = float(np.mean(np.abs(rr_diff)>50))*100 if len(rr_diff)>0 else 0.
    f['hrv_mean_rr'] = float(np.mean(rr))
    f['hrv_cv_rr']   = f['hrv_sdnn'] / f['hrv_mean_rr'] if f['hrv_mean_rr']>1e-6 else 0.
    return f

def extract_features(label_df, sensor_df, pid_enc_map):
    sensor_by_pid = {pid: grp.sort_values('timestamp').reset_index(drop=True)
                     for pid, grp in sensor_df.groupby('pid')}
    rows = []
    for _, lrow in label_df.iterrows():
        pid = lrow['pid']; ts = float(lrow['timestamp']); lid = lrow['id']
        feat = {'id': lid}
        sg = sensor_by_pid.get(pid)
        if sg is None: rows.append(feat); continue
        ta = sg['timestamp'].values
        wa  = sg.loc[(ta>=ts-WINDOW_MS)&(ta<=ts),           SENSOR_COLS]
        wf  = sg.loc[(ta>=ts-WINDOW_MS)&(ta<ts-HALF_MS),    SENSOR_COLS]
        wl  = sg.loc[(ta>=ts-HALF_MS)  &(ta<=ts),           SENSOR_COLS]
        wt1 = sg.loc[(ta>=ts-WINDOW_MS)&(ta<ts-2*THIRD_MS), SENSOR_COLS]
        wt3 = sg.loc[(ta>=ts-THIRD_MS) &(ta<=ts),           SENSOR_COLS]

        for c in SENSOR_COLS:
            v   = wa[c].dropna().values.astype(float)
            vf  = wf[c].dropna().values.astype(float)
            vl  = wl[c].dropna().values.astype(float)
            vt1 = wt1[c].dropna().values.astype(float)
            vt3 = wt3[c].dropna().values.astype(float)
            if len(v) == 0:
                for s in ['mean','std','min','max','median','skew','kurt','range',
                          'q25','q75','iqr','delta','slope','t1_mean','t3_mean','t3t1']:
                    feat[f'{c}_{s}'] = np.nan
                continue
            feat[f'{c}_mean']   = float(np.mean(v))
            feat[f'{c}_std']    = float(np.std(v))
            feat[f'{c}_min']    = float(np.min(v))
            feat[f'{c}_max']    = float(np.max(v))
            feat[f'{c}_median'] = float(np.median(v))
            feat[f'{c}_skew']   = float(spstats.skew(v))     if len(v)>2 else 0.
            feat[f'{c}_kurt']   = float(spstats.kurtosis(v)) if len(v)>2 else 0.
            feat[f'{c}_range']  = float(np.max(v)-np.min(v))
            feat[f'{c}_q25']    = float(np.percentile(v,25))
            feat[f'{c}_q75']    = float(np.percentile(v,75))
            feat[f'{c}_iqr']    = float(np.percentile(v,75)-np.percentile(v,25))
            feat[f'{c}_delta']  = float(np.mean(vl)-np.mean(vf)) if len(vf)>0 and len(vl)>0 else 0.
            feat[f'{c}_slope']  = float(np.polyfit(np.linspace(0,1,len(v)),v,1)[0]) if len(v)>2 else 0.
            feat[f'{c}_t1_mean']= float(np.mean(vt1)) if len(vt1)>0 else float(np.mean(v))
            feat[f'{c}_t3_mean']= float(np.mean(vt3)) if len(vt3)>0 else float(np.mean(v))
            feat[f'{c}_t3t1']   = feat[f'{c}_t3_mean'] - feat[f'{c}_t1_mean']

        # Accel magnitude
        ax=wa['accel_x'].values; ay=wa['accel_y'].values; az=wa['accel_z'].values
        if len(ax)>0:
            mag = np.sqrt(ax**2+ay**2+az**2)
            feat['accel_mag_mean']=float(np.mean(mag))
            feat['accel_mag_std'] =float(np.std(mag))
            feat['accel_mag_max'] =float(np.max(mag))
        else:
            feat['accel_mag_mean']=feat['accel_mag_std']=feat['accel_mag_max']=np.nan

        # Time-domain HRV only (no frequency domain — too noisy from 1Hz BPM)
        feat.update(hrv_time_domain(wa['heart_rate']))

        # pid_enc
        feat['pid_enc'] = pid_enc_map.get(pid, -1)
        rows.append(feat)
    return pd.DataFrame(rows).set_index('id')

train_pid_map = {p: i for i, p in enumerate(TRAIN_LABEL['pid'].unique())}

print('Extracting train features...')
train_features = extract_features(TRAIN_LABEL, TRAIN_DATA, train_pid_map)
print(f'  shape: {train_features.shape}')
print('Extracting test features...')
test_features = extract_features(TEST_LABEL, TEST_DATA, train_pid_map)
print(f'  shape: {test_features.shape}')


Extracting train features...
  shape: (815, 106)
Extracting test features...
  shape: (1028, 106)


In [4]:
tli    = TRAIN_LABEL.set_index('id')
y      = tli['stress'].astype(int)
groups = tli['pid']

imputer    = SimpleImputer(strategy='median')
X_imp      = pd.DataFrame(imputer.fit_transform(train_features),
                           columns=train_features.columns, index=train_features.index)
X_test_imp = pd.DataFrame(imputer.transform(test_features),
                           columns=test_features.columns, index=test_features.index)

counts = Counter(y); total = len(y); n_cls = len(counts)

# Reduced class 1 weight to prevent overconfidence (was 4.12, now 2.5)
class_weights = {
    0: total / (n_cls * counts[0]),       # ~1.68 — keep
    1: min(total / (n_cls * counts[1]), 2.5),  # cap at 2.5 (was 4.12)
    2: total / (n_cls * counts[2]),       # ~0.46 — keep
}
sample_weights = np.array([class_weights[yi] for yi in y])

print('X_imp shape     :', X_imp.shape)
print('X_test_imp shape:', X_test_imp.shape)
print('Class distribution:', dict(counts))
print('Class weights (capped):', {k: round(v,3) for k,v in class_weights.items()})
print()
print('Training prior (class proportions):')
train_prior = np.array([counts[i]/total for i in range(3)])
print('  ', {i: round(train_prior[i],3) for i in range(3)})


X_imp shape     : (815, 106)
X_test_imp shape: (1028, 106)
Class distribution: {1: 66, 0: 162, 2: 587}
Class weights (capped): {0: 1.677, 1: 2.5, 2: 0.463}

Training prior (class proportions):
   {0: np.float64(0.199), 1: np.float64(0.081), 2: np.float64(0.72)}


In [5]:
LGBM_PARAMS = dict(
    n_estimators      = 1000,
    learning_rate     = 0.02,
    num_leaves        = 127,
    max_depth         = -1,
    min_child_samples = 5,
    subsample         = 0.6,
    colsample_bytree  = 0.6,
    reg_alpha         = 0.3,
    reg_lambda        = 0.3,
    class_weight      = 'balanced',
    objective         = 'multiclass',
    num_class         = 3,
    n_jobs            = -1,
    verbose           = -1,
)

print('=== LOPO CV — v7b (absolute + change + time-domain HRV + pid_enc) ===')
logo = LeaveOneGroupOut()
lopo_scores = []

for tr_idx, val_idx in logo.split(X_imp, y, groups):
    pid_val = groups.iloc[val_idx[0]]
    y_val   = y.iloc[val_idx]
    if len(y_val.unique()) < 2:
        print(f'  Skip {pid_val}: only 1 class'); continue

    m = lgb.LGBMClassifier(**{**LGBM_PARAMS, 'random_state': 42})
    m.fit(X_imp.iloc[tr_idx], y.iloc[tr_idx],
          sample_weight = sample_weights[tr_idx],
          eval_set      = [(X_imp.iloc[val_idx], y_val)],
          callbacks     = [lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])

    sc = balanced_accuracy_score(y_val, m.predict(X_imp.iloc[val_idx]))
    print(f'  Leave out {pid_val}: {sc:.4f}  (n={len(val_idx)}, classes={sorted(y_val.unique())})')
    lopo_scores.append(sc)

print(f'\nv7b LOPO = {np.mean(lopo_scores):.4f} +/- {np.std(lopo_scores):.4f}')
print('v7  LOPO = 0.5123  |  v4 best = 0.4288')


=== LOPO CV — v7b (absolute + change + time-domain HRV + pid_enc) ===
  Leave out 43JW: 0.5000  (n=93, classes=[np.int64(0), np.int64(2)])
  Leave out C8Q6: 0.4965  (n=152, classes=[np.int64(0), np.int64(2)])
  Leave out DT5C: 0.5460  (n=90, classes=[np.int64(0), np.int64(1), np.int64(2)])
  Leave out F1ZM: 0.5000  (n=137, classes=[np.int64(1), np.int64(2)])
  Leave out HDS9: 0.6517  (n=135, classes=[np.int64(0), np.int64(2)])
  Leave out P4DZ: 0.2826  (n=144, classes=[np.int64(0), np.int64(1), np.int64(2)])
  Leave out TPQI: 0.6141  (n=64, classes=[np.int64(0), np.int64(2)])

v7b LOPO = 0.5130 +/- 0.1097
v7  LOPO = 0.5123  |  v4 best = 0.4288


In [6]:
print('=== Final ensemble: 3 seeds x 5 folds ===')
SEEDS = [42, 7, 123]
all_test_proba = []
all_cv_scores  = []

for seed in SEEDS:
    skf        = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    seed_proba = np.zeros((len(X_test_imp), 3))
    fold_scores = []

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_imp, y)):
        X_tr = X_imp.iloc[tr_idx]; y_tr = y.iloc[tr_idx]
        X_va = X_imp.iloc[val_idx]; y_va = y.iloc[val_idx]
        sw_tr = sample_weights[tr_idx]

        m = lgb.LGBMClassifier(**{**LGBM_PARAMS, 'random_state': seed})
        m.fit(X_tr, y_tr,
              sample_weight = sw_tr,
              eval_set      = [(X_va, y_va)],
              callbacks     = [lgb.early_stopping(100, verbose=False), lgb.log_evaluation(-1)])

        sc = balanced_accuracy_score(y_va, m.predict(X_va))
        fold_scores.append(sc)
        seed_proba += m.predict_proba(X_test_imp)
        print(f'  Seed {seed} Fold {fold+1}: val BA = {sc:.4f}')

    seed_proba /= 5
    all_test_proba.append(seed_proba)
    all_cv_scores.append(np.mean(fold_scores))
    print(f'  Seed {seed} mean CV = {np.mean(fold_scores):.4f}')

print(f'\nEnsemble CV = {np.mean(all_cv_scores):.4f}')

# Raw soft-vote probabilities
raw_proba = np.mean(all_test_proba, axis=0)

# Prior calibration: multiply raw proba by training prior, then renormalize
# This pulls class 1 back toward its 8% training frequency
train_prior = np.array([counts[i]/total for i in range(3)])
calibrated_proba = raw_proba * train_prior
calibrated_proba = calibrated_proba / calibrated_proba.sum(axis=1, keepdims=True)

final_preds = np.argmax(calibrated_proba, axis=1).astype(int)

print()
print('Raw prediction distribution (before calibration):')
for u, cnt in zip(*np.unique(np.argmax(raw_proba,1), return_counts=True)):
    print(f'  class {u}: {cnt}  ({cnt/len(final_preds)*100:.1f}%)')

print()
print('Calibrated prediction distribution (after prior rescaling):')
for u, cnt in zip(*np.unique(final_preds, return_counts=True)):
    print(f'  class {u}: {cnt}  ({cnt/len(final_preds)*100:.1f}%)')

print()
print('Train distribution for reference:')
for cls, cnt in sorted(Counter(y).items()):
    print(f'  class {cls}: {cnt}  ({cnt/len(y)*100:.1f}%)')


=== Final ensemble: 3 seeds x 5 folds ===
  Seed 42 Fold 1: val BA = 0.8414
  Seed 42 Fold 2: val BA = 0.7679
  Seed 42 Fold 3: val BA = 0.8314
  Seed 42 Fold 4: val BA = 0.7622
  Seed 42 Fold 5: val BA = 0.8495
  Seed 42 mean CV = 0.8105
  Seed 7 Fold 1: val BA = 0.8179
  Seed 7 Fold 2: val BA = 0.8094
  Seed 7 Fold 3: val BA = 0.8327
  Seed 7 Fold 4: val BA = 0.8341
  Seed 7 Fold 5: val BA = 0.7982
  Seed 7 mean CV = 0.8184
  Seed 123 Fold 1: val BA = 0.8614
  Seed 123 Fold 2: val BA = 0.6913
  Seed 123 Fold 3: val BA = 0.8809
  Seed 123 Fold 4: val BA = 0.8220
  Seed 123 Fold 5: val BA = 0.7724
  Seed 123 mean CV = 0.8056

Ensemble CV = 0.8115

Raw prediction distribution (before calibration):
  class 0: 400  (38.9%)
  class 1: 479  (46.6%)
  class 2: 149  (14.5%)

Calibrated prediction distribution (after prior rescaling):
  class 0: 364  (35.4%)
  class 1: 164  (16.0%)
  class 2: 500  (48.6%)

Train distribution for reference:
  class 0: 162  (19.9%)
  class 1: 66  (8.1%)
  class 

In [7]:
submission = pd.DataFrame({'id': TEST_LABEL['id'].values, 'stress': final_preds})
submission.to_csv('submission_v7b.csv', index=False)
print('submission_v7b.csv saved!')
print(submission.head(10))


submission_v7b.csv saved!
     id  stress
0  1227       2
1  1228       0
2  1229       2
3  1230       2
4  1231       2
5  1232       2
6  1233       0
7  1234       2
8  1235       0
9  1236       0
